# Compatifi V5A — Assistant Objective Model Test

Tests `V5A_Final_Merged_Model`.

Input: Domain + Relationship + Conversation  
Output: `primary_objective`, `secondary_objective`, `priority`, `reason`

Includes strict JSON validation, extra-field detection, controlled labels, Qwen thinking suppression, and raw/cleaned output.

In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

/home/ubuntu/V5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_PATH = "./V5A_Final_Merged_Model"
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 180

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

GPU_NAME = torch.cuda.get_device_name(0)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("=" * 80)
print("COMPATIFI V5A TEST")
print("=" * 80)
print(f"GPU   : {GPU_NAME}")
print(f"MODEL : {MODEL_PATH}")
print(f"DTYPE : {COMPUTE_DTYPE}")
print("=" * 80)

COMPATIFI V5A TEST
GPU   : NVIDIA L40S
MODEL : ./V5A_Final_Merged_Model
DTYPE : torch.bfloat16


In [3]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

print("Loading V5A merged model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    trust_remote_code=True,
)
model.eval()
print("V5A merged model loaded successfully.")

Loading tokenizer...
Loading V5A merged model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.59s/it]

V5A merged model loaded successfully.


In [4]:
SYSTEM_PROMPT = """
You are Compatifi V5A.

Your task is to predict what the assistant should try to accomplish
in the current conversation.

Predict the assistant's objective, NOT the user's long-term goal.

Identify exactly:
- primary_objective
- secondary_objective
- priority
- reason

Always provide one primary_objective.
If there is no meaningful secondary objective, use exactly: "None".

The reason must be short and based only on the conversation.

STRICT OUTPUT RULES:
Return exactly ONE JSON object.
The JSON object MUST contain exactly these four keys:
"primary_objective", "secondary_objective", "priority", "reason"

Do NOT output any other keys.
Do NOT output <think> or </think>.
Do NOT output Markdown or ```json.
Do NOT output explanations before or after the JSON.

Allowed primary_objective values:
"Emotional Support"
"Reduce Anxiety"
"Solve Problem"
"Decision Support"
"Planning Assistance"
"Motivation"
"Information Sharing"

Allowed priority values:
"High", "Medium", "Low"

Do NOT generate the final reply.
Do NOT give advice directly.
Do NOT generate conversation messages.
Do NOT extract memories.
Do NOT predict personality.
Do NOT predict long-term goals.
Do NOT invent information.
"""

In [27]:
test_cases = [
#     {
#         "name": "Interview Anxiety",
#         "domain": "relationship",
#         "relationship": "Friend",
#         "conversation": """Friend: You seem worried lately.
# User: I have my final interview tomorrow and I'm really nervous.
# Friend: You've prepared for weeks.
# User: I know, but I keep thinking I'll mess it up."""
#     },
#     {
#         "name": "Family Decision",
#         "domain": "relationship",
#         "relationship": "Partner",
#         "conversation": """Partner: We need to decide where to spend Christmas this year.
# User: I don't know whether we should visit your parents or mine."""
#     },
#     {
#         "name": "Resume Help",
#         "domain": "career",
#         "relationship": "Career Coach",
#         "conversation": """Career Coach: How is your resume coming along?
# User: I'm struggling to describe my leadership experience."""
#     },
#     {
#         "name": "Bad Day",
#         "domain": "relationship",
#         "relationship": "Friend",
#         "conversation": """Friend: How was your day?
# User: Honestly, it was terrible.
# Friend: What happened?
# User: Everything went wrong at work and I'm completely exhausted."""
#     },
#     {
#         "name": "Work Problem",
#         "domain": "work",
#         "relationship": "Colleague",
#         "conversation": """Colleague: The project deadline was moved to Friday.
# User: That's a problem. We still have three major tasks unfinished.
# Colleague: We need to figure out how to finish everything."""
#     },
#     {
#         "name": "Weekend Planning",
#         "domain": "relationship",
#         "relationship": "Friend",
#         "conversation": """Friend: We haven't done anything together for weeks.
# User: Yeah, we've both been busy.
# Friend: We should do something this weekend.
# User: Definitely. Let's plan something."""
#     },
#     {
#         "name": "Casual Conversation",
#         "domain": "relationship",
#         "relationship": "Friend",
#         "conversation": """Friend: What are you doing?
# User: Just watching TV.
# Friend: Anything good?
# User: Just a comedy."""
#     },
#     {
#         "name": "Learning Progress",
#         "domain": "personal",
#         "relationship": "Mentor",
#         "conversation": """Mentor: How is your programming practice going?
# User: It's difficult, but I've been practicing every day.
# Mentor: That's good progress.
# User: Sometimes I feel like I'm not improving fast enough."""
#     },
#     {
#         "name": "Friendship Conflict",
#         "domain": "relationship",
#         "relationship": "Friend",
#         "conversation": """Friend: You didn't reply to me yesterday.
# User: I was busy.
# Friend: You always say that.
# User: I'm sorry. I didn't mean to ignore you."""
#     },
#     {
#         "name": "Simple Question",
#         "domain": "general",
#         "relationship": "Friend",
#         "conversation": """Friend: Do you know what time the movie starts?
# User: I think it's at eight.
# Friend: Are you sure?
# User: Let me check."""
#     },
#     {
#         "name": "Low Motivation",
#         "domain": "personal",
#         "relationship": "Friend",
#         "conversation": """Friend: Have you started your project yet?
# User: No, I keep putting it off.
# Friend: What's stopping you?
# User: I just don't feel motivated."""
#     },
#     {
#         "name": "Relationship Problem",
#         "domain": "relationship",
#         "relationship": "Partner",
#         "conversation": """Partner: I feel like we've been arguing a lot lately.
# User: I don't want things to keep going this way.
# Partner: Me neither.
# User: I think we need to figure out what's causing these arguments."""
#     },




    {
        "name": "V4A Replay Retention Test",
        "domain": "relationship",
        "relationship": "Friend",

        "instruction": "Extract long-term memories",

        "conversation": """Friend: How have you been spending your evenings lately?
User: I've been practicing photography almost every evening.
Friend: Still working toward your exhibition?
User: Yes. I plan to hold my first photography exhibition next year.
Friend: You seem to really enjoy landscape photography.
User: I do. I've been focusing mostly on landscape photography for the past year."""
    }
]

In [28]:
def generate_v5a(test_case):
    user_content = f"""Domain: {test_case["domain"]}

Relationship: {test_case["relationship"]}

Instruction: Predict the assistant objective

Conversation:
{test_case["conversation"].strip()}"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

In [29]:
def clean_v5a_response(response):
    response = response.strip()

    if "<think>" in response:
        response = response.split("<think>", 1)[1]
    if "</think>" in response:
        response = response.split("</think>", 1)[1]

    response = response.strip()

    if response.startswith("```json"):
        response = response[7:]
    elif response.startswith("```"):
        response = response[3:]

    if response.endswith("```"):
        response = response[:-3]

    return response.strip()


REQUIRED_FIELDS = {
    "primary_objective",
    "secondary_objective",
    "priority",
    "reason",
}

ALLOWED_PRIMARY_OBJECTIVES = {
    "Emotional Support",
    "Reduce Anxiety",
    "Solve Problem",
    "Decision Support",
    "Planning Assistance",
    "Motivation",
    "Information Sharing",
}

ALLOWED_PRIORITIES = {"High", "Medium", "Low"}


def validate_v5a_output(response):
    try:
        data = json.loads(response)
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}", None

    if not isinstance(data, dict):
        return False, "Output is not a JSON object", None

    missing = REQUIRED_FIELDS - set(data.keys())
    if missing:
        return False, f"Missing fields: {sorted(missing)}", data

    extra = set(data.keys()) - REQUIRED_FIELDS
    if extra:
        return False, f"Unexpected fields: {sorted(extra)}", data

    if not isinstance(data["primary_objective"], str) or not data["primary_objective"].strip():
        return False, "primary_objective is empty or invalid", data

    if data["primary_objective"] not in ALLOWED_PRIMARY_OBJECTIVES:
        return False, f"Invalid primary_objective: {data['primary_objective']}", data

    if not isinstance(data["secondary_objective"], str) or not data["secondary_objective"].strip():
        return False, "secondary_objective is empty or invalid", data

    if data["priority"] not in ALLOWED_PRIORITIES:
        return False, f"Invalid priority: {data['priority']}", data

    if not isinstance(data["reason"], str) or not data["reason"].strip():
        return False, "reason is empty or invalid", data

    return True, "Valid V5A output", data

In [30]:
print("=" * 80)
print("STARTING V5A TESTS")
print("=" * 80)

valid_count = 0
raw_json_count = 0
results = []

for index, test_case in enumerate(test_cases, 1):
    print()
    print("=" * 80)
    print(f"TEST {index}: {test_case['name']}")
    print("=" * 80)
    print(f"DOMAIN: {test_case['domain']}")
    print(f"RELATIONSHIP: {test_case['relationship']}")
    print()
    print("CONVERSATION:")
    print(test_case["conversation"].strip())

    raw_response = generate_v5a(test_case)

    print()
    print("-" * 80)
    print("RAW OUTPUT")
    print("-" * 80)
    print(raw_response)

    try:
        json.loads(raw_response)
        raw_json_count += 1
        raw_ok = True
    except json.JSONDecodeError:
        raw_ok = False

    cleaned_response = clean_v5a_response(raw_response)

    print()
    print("-" * 80)
    print("CLEANED OUTPUT")
    print("-" * 80)
    print(cleaned_response)

    is_valid, message, parsed = validate_v5a_output(cleaned_response)

    print()
    print("VALIDATION")
    if is_valid:
        valid_count += 1
        print("✅", message)
        print("Primary Objective   :", parsed["primary_objective"])
        print("Secondary Objective :", parsed["secondary_objective"])
        print("Priority            :", parsed["priority"])
        print("Reason              :", parsed["reason"])
    else:
        print("❌", message)

    results.append({
        "test": index,
        "name": test_case["name"],
        "raw_json": raw_ok,
        "valid_v5a": is_valid,
        "message": message,
    })

total = len(test_cases)

print()
print("=" * 80)
print("V5A TEST SUMMARY")
print("=" * 80)
print(f"Total tests        : {total}")
print(f"Raw JSON valid     : {raw_json_count}")
print(f"Clean V5A valid    : {valid_count}")
print(f"Invalid            : {total - valid_count}")
print(f"Raw JSON success   : {raw_json_count / total * 100:.1f}%")
print(f"V5A schema success : {valid_count / total * 100:.1f}%")
print("=" * 80)
print("V5A TEST COMPLETE")
print("=" * 80)

STARTING V5A TESTS

TEST 1: V4A Replay Retention Test
DOMAIN: relationship
RELATIONSHIP: Friend

CONVERSATION:
Friend: How have you been spending your evenings lately?
User: I've been practicing photography almost every evening.
Friend: Still working toward your exhibition?
User: Yes. I plan to hold my first photography exhibition next year.
Friend: You seem to really enjoy landscape photography.
User: I do. I've been focusing mostly on landscape photography for the past year.


/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



--------------------------------------------------------------------------------
RAW OUTPUT
--------------------------------------------------------------------------------
{"primary_objective":"Motivation","secondary_objective":"None","priority":"Medium","reason":"The user is actively pursuing a photography exhibition and needs ongoing motivation to stay on track.","memories":null}

--------------------------------------------------------------------------------
CLEANED OUTPUT
--------------------------------------------------------------------------------
{"primary_objective":"Motivation","secondary_objective":"None","priority":"Medium","reason":"The user is actively pursuing a photography exhibition and needs ongoing motivation to stay on track.","memories":null}

VALIDATION
❌ Unexpected fields: ['memories']

V5A TEST SUMMARY
Total tests        : 1
Raw JSON valid     : 1
Clean V5A valid    : 0
Invalid            : 1
Raw JSON success   : 100.0%
V5A schema success : 0.0%
V5A TEST COM